Question 1 — Does temperature significantly affect crop yield?

In [1]:
import pandas as pd 
from scipy.stats import pearsonr 

df = pd.read_csv('../data/climate_change_impact_on_agriculture_2024.csv')
correlation, p_value = pearsonr(df['Average_Temperature_C'], df['Crop_Yield_MT_per_HA'])

def strength_label(correlation):
    abs_r = abs(correlation)
    if abs_r >= 0.9:
        return "Very Strong"
    elif abs_r >= 0.7:
        return "Strong"
    elif abs_r >= 0.5:
        return "Moderate"
    elif abs_r >= 0.3:
        return "Weak"
    else:
        return "Very Weak"

label = strength_label(correlation)
print(f"Correlation between Temperature and Crop Yield: {correlation:.2f}")
print(f"The P-value is: {p_value:.4f}")
print(f"The Strength label: {label}")

Correlation between Temperature and Crop Yield: 0.26
The P-value is: 0.0000
The Strength label: Very Weak


Question 2 — Is there a significant difference in crop yield between countries WITH adaptation strategies vs WITHOUT?

In [2]:
from scipy import stats 

adapted = df[df["Adaptation_Strategies"] != "No Adaptation"]["Crop_Yield_MT_per_HA"]
no_adapt = df[df["Adaptation_Strategies"] == "No Adaptation"]["Crop_Yield_MT_per_HA"]
# Run t-test
t_stat, p_value = stats.ttest_ind(adapted, no_adapt, equal_var=False)

def p_value_res(p_value):
    if p_value < 0.001:
        return "Very strong evidence against null (p < 0.001)"
    elif p_value < 0.05:
        return "Significant — reject null hypothesis (p < 0.05)"
    else:
        return "Not significant — fail to reject null (p ≥ 0.05)"

def t_stat_res(t_stat):
    if 0 <= t_stat <= 0.5:
        return "Very small to negligible difference."
    elif 0.5 < t_stat <= 1.0:
        return "Small difference; may be significant only with huge samples."
    elif 1.0 < t_stat <= 2.0:
        return "Small to moderate; significance depends on df and α."
    elif 2.0 < t_stat <= 3.0:
        return "Moderate; often significant if df is moderate."
    elif t_stat > 3.0:
        return "Large difference; almost always significant."

t_interpret = t_stat_res(t_stat)
p_interpret = p_value_res(p_value)

print(f"The t_test = : {t_stat:.2f}, {t_interpret}")
print(f"The p_value = : {p_value:.2f}, {p_interpret}")



The t_test = : 0.10, Very small to negligible difference.
The p_value = : 0.92, Not significant — fail to reject null (p ≥ 0.05)


Question 3 — Which climate factor correlates most strongly with crop yield?

In [5]:
factors = ["Average_Temperature_C", "Total_Precipitation_mm", 
           "CO2_Emissions_MT", "Soil_Health_Index",
           "Extreme_Weather_Events"]
# Calculate pearsonr for each factor vs Crop_Yield_MT_per_HA
def strength_label(correlation):
    abs_r = abs(correlation)
    if abs_r < 0.3:
        return "Weak"
    elif abs_r < 0.7:
        return "Moderate"
    else:
        return "Strong"


results = {}
for factor in factors:
    correlation, p_value = pearsonr(df[factor], df['Crop_Yield_MT_per_HA'])
    results[factor] = {
        'Correlation': round(correlation, 4),
        'P-value': round(p_value, 4),
        'Significant': strength_label(correlation)
        }

summary_df["Abs_Correlation"] = summary_df["Correlation"].abs()
summary_df = summary_df.sort_values("Abs_Correlation", ascending=False)\
                        .drop(columns="Abs_Correlation")\
                        .reset_index(drop=True)

summary_df.index = summary_df.index + 1 
summary_df.index.name = "Rank"

print("====Climate Factors vs Crop Yield====")
print(summary_df.to_string())
print(f"\nStrongest factor: {summary_df.iloc[0]["Factor"]}")
print(f"Correlation: {summary_df.iloc[0]['Correlation']}")    


====Climate Factors vs Crop Yield====
                      Factor  Correlation  P-value Significant
Rank                                                          
1      Average_Temperature_C       0.2638   0.0000        Weak
2           CO2_Emissions_MT      -0.0899   0.0000        Weak
3     Total_Precipitation_mm       0.0297   0.0029        Weak
4          Soil_Health_Index      -0.0057   0.5692        Weak
5     Extreme_Weather_Events      -0.0051   0.6105        Weak

Strongest factor: Average_Temperature_C
Correlation: 0.2638


Question 4 — For Central Asia crop types specifically — does irrigation access significantly affect yield?

In [4]:
# Use Crop types
ca_crops = ["Wheat", "Cotton", "Rice", "Barley"]
# Split into high irrigation (>50%) vs low irrigation (<50%)
crop_df = df[df['Crop_Type'].isin(ca_crops)]

high_irr = crop_df[crop_df['Irrigation_Access_%'] > 50]['Crop_Yield_MT_per_HA']
low_irr = crop_df[crop_df['Irrigation_Access_%'] < 50]['Crop_Yield_MT_per_HA']

# Run t-test
t_stat, p_value = stats.ttest_ind(high_irr, low_irr, equal_var=False)

# Write a 3-sentence conclusion
print(f"T-test: {round(t_stat, 4)}, P-value: {round(p_value, 4)}")
if p_value < 0.05:
    print("Significant difference – irrigation access affects yield.")
else:
    print("No significant difference.")

T-test: -0.0181, P-value: 0.9855
No significant difference.


Conclusion:

Analysis of 10,000 records spanning 1990–2024 reveals that climate 
factors have limited linear predictive power over crop yield in this 
dataset. Temperature shows the strongest correlation with yield 
(r = 0.26, p < 0.001), though this relationship is classified as weak, 
suggesting non-linear dynamics or confounding variables may be at play.

Adaptation strategies — including crop rotation, water management and 
drought-resistant varieties — showed no statistically significant effect 
on yield compared to non-adapted fields (t = 0.10, p = 0.92). This 
counter-intuitive finding may reflect the dataset's synthetic nature or 
masking effects from regional variability.

Among Central Asia-relevant crops (wheat, cotton, rice, barley), 
irrigation access above 50% did not significantly differentiate yield 
outcomes (t = -0.02, p = 0.99), suggesting that irrigation alone is 
insufficient without complementary agronomic inputs.

Overall, no single climate factor demonstrates strong linear correlation 
with crop yield, indicating that agricultural productivity is driven by 
complex multivariate interactions rather than individual climate variables.